# GridPulse BR — Regulatory Limits Source Profiling

## Objective

Profile the ANEEL regulatory continuity limits dataset before integrating it
into the GridPulse BR data product.

This source will later be used to compare actual DEC and FEC values against
their corresponding regulatory limits.

## Questions to answer

- What is the source schema?
- What is the dataset grain?
- Which indicators are available?
- Which years are covered?
- Are there duplicate keys?
- Are there null values?
- How do the limits relate to consumer-unit sets?
- Can the dataset be safely joined to the existing Gold product?

In [0]:
import requests
from pathlib import Path

SOURCE_URL = (
    "https://dadosabertos.aneel.gov.br/dataset/"
    "d5f0712e-62f6-4736-8dff-9991f10758a7/resource/"
    "fd69e1dd-fd66-4269-b60c-cc0b7eb221b4/download/"
    "indicadores-continuidade-coletivos-limite.csv"
)

LANDING_DIR = (
    "/Volumes/workspace/gridpulse/landing/aneel/"
    "continuity_limits"
)

TARGET_FILE = (
    f"{LANDING_DIR}/"
    "indicadores-continuidade-coletivos-limite.csv"
)

print(f"Source URL  : {SOURCE_URL}")
print(f"Target file : {TARGET_FILE}")

In [0]:
dbutils.fs.mkdirs(LANDING_DIR)

print(f"Landing directory ready: {LANDING_DIR}")

In [0]:
response = requests.head(
    SOURCE_URL,
    allow_redirects=True,
    timeout=30
)

print(f"Status code  : {response.status_code}")
print(f"Content type : {response.headers.get('content-type')}")
print(f"Content size : {response.headers.get('content-length')}")
print(f"Final URL    : {response.url}")

response.raise_for_status()

In [0]:
response = requests.get(
    SOURCE_URL,
    allow_redirects=True,
    timeout=120
)

response.raise_for_status()

with open(TARGET_FILE, "wb") as file:
    file.write(response.content)

print("Download completed.")
print(f"Bytes written: {len(response.content):,}")

In [0]:
files = dbutils.fs.ls(LANDING_DIR)

display(files)

In [0]:
with open(TARGET_FILE, "r", encoding="utf-8-sig") as file:
    for _ in range(5):
        print(file.readline().rstrip())

In [0]:
## 1. Read regulatory limits source

Read the ANEEL regulatory limits CSV using its native delimiter and decimal
format before profiling the source structure and business grain.

In [0]:
df_limits_raw = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .option("inferSchema", "true")
    .option("locale", "pt-BR")
    .csv(TARGET_FILE)
)

limits_row_count = df_limits_raw.count()

print(f"Rows    : {limits_row_count:,}")
print(f"Columns : {len(df_limits_raw.columns)}")

df_limits_raw.printSchema()

## 2. Profile temporal and indicator coverage

In [0]:
import pyspark.sql.functions as F

display(
    df_limits_raw
    .groupBy("SigIndicador")
    .agg(
        F.count("*").alias("rows"),
        F.min("AnoLimiteQualidade").alias("min_year"),
        F.max("AnoLimiteQualidade").alias("max_year"),
        F.countDistinct("AnoLimiteQualidade").alias("distinct_years")
    )
    .orderBy("SigIndicador")
)

In [0]:
display(
    df_limits_raw
    .groupBy("AnoLimiteQualidade")
    .count()
    .orderBy("AnoLimiteQualidade")
)

## 3. Validate candidate business grain

Test whether each consumer-unit set has a single regulatory limit for each
indicator and quality-limit year.

In [0]:
LIMIT_KEY = [
    "IdeConjUndConsumidoras",
    "SigIndicador",
    "AnoLimiteQualidade"
]

limit_key_profile = (
    df_limits_raw
    .groupBy(*LIMIT_KEY)
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("VlrLimite").alias("distinct_limit_values"),
        F.countDistinct("SigAgente").alias("agent_count"),
        F.countDistinct("NumCNPJ").alias("cnpj_count")
    )
)

duplicate_limit_keys = (
    limit_key_profile
    .filter(F.col("row_count") > 1)
)

duplicate_limit_key_count = duplicate_limit_keys.count()

print(f"Duplicate candidate keys : {duplicate_limit_key_count:,}")

display(
    duplicate_limit_keys
    .groupBy(
        "row_count",
        "distinct_limit_values",
        "agent_count",
        "cnpj_count"
    )
    .count()
    .orderBy("row_count")
)

In [0]:
null_profile = df_limits_raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_limits_raw.columns
])

display(null_profile)

In [0]:
print(
    "Consumer sets:",
    df_limits_raw.select("IdeConjUndConsumidoras").distinct().count()
)

print(
    "Distributors:",
    df_limits_raw.select("SigAgente").distinct().count()
)

print(
    "Indicators:",
    df_limits_raw.select("SigIndicador").distinct().count()
)

## 4. Investigate duplicate candidate keys

Inspect candidate-key collisions before defining the regulatory-limit grain.

These records share the same consumer-unit set, indicator and limit year,
but are associated with different distributors.

In [0]:
duplicate_key_values = (
    duplicate_limit_keys
    .select(*LIMIT_KEY)
)

duplicate_details = (
    df_limits_raw.alias("src")
    .join(
        duplicate_key_values.alias("dup"),
        on=LIMIT_KEY,
        how="inner"
    )
    .select(
        "IdeConjUndConsumidoras",
        "DscConjUndConsumidoras",
        "SigIndicador",
        "AnoLimiteQualidade",
        "VlrLimite",
        "SigAgente",
        "NumCNPJ",
        "DatGeracaoConjuntoDados"
    )
    .orderBy(
        "IdeConjUndConsumidoras",
        "AnoLimiteQualidade",
        "SigIndicador",
        "SigAgente"
    )
)

display(duplicate_details)

In [0]:
display(
    duplicate_details
    .groupBy(
        "IdeConjUndConsumidoras",
        "DscConjUndConsumidoras",
        "AnoLimiteQualidade",
        "SigIndicador"
    )
    .agg(
        F.collect_set("SigAgente").alias("agents"),
        F.collect_set("NumCNPJ").alias("cnpjs"),
        F.collect_set("VlrLimite").alias("limits"),
        F.collect_set("DatGeracaoConjuntoDados").alias("generation_dates")
    )
    .orderBy(
        "IdeConjUndConsumidoras",
        "AnoLimiteQualidade"
    )
)

In [0]:
df_limits_typed = (
    df_limits_raw
    .withColumn(
        "VlrLimite_num",
        F.regexp_replace(
            F.col("VlrLimite"),
            ",",
            "."
        ).cast("double")
    )
)

In [0]:
limit_value_profile = (
    df_limits_typed
    .agg(
        F.count("*").alias("rows"),
        F.sum(
            F.when(F.col("VlrLimite_num").isNull(), 1).otherwise(0)
        ).alias("invalid_limit_values"),
        F.min("VlrLimite_num").alias("min_limit"),
        F.max("VlrLimite_num").alias("max_limit")
    )
)

display(limit_value_profile)

## Profiling conclusion

The regulatory-limits source contains 263,401 records covering DEC and FEC
limits from 1990/1996 through 2032.

The candidate business grain is:

- consumer-unit set
- indicator
- quality-limit year

Twelve candidate-key collisions were identified. All occur in 2010 and are
associated with the same consumer-unit set, indicator, year and regulatory
limit, but with two different distributor/CNPJ combinations (EBO and EPB).

Because the regulatory limit itself is identical in each collision, distributor
ownership is treated as a descriptive attribute rather than part of the
regulatory-limit business key.

The Silver layer will therefore deduplicate these records using the regulatory
business key while preserving one canonical observation per limit.